# 02 — Auditer les partitions temporelles

Notebook Google Colab pour le projet Bug Predictor. Exécuter les cellules dans l’ordre.

## Objectif
Auditer les trois partitions temporelles `train`, `val` et `test` de la famille `line_bug_prediction_splits`. Le notebook lit par lots pour limiter la mémoire, contrôle le schéma, les valeurs nulles et la cohérence des numéros de lignes. Il calcule aussi les recouvrements exacts de contenu et les recouvrements `(repo, filepath)` entre splits au moyen d’une base SQLite temporaire. Le split test n’est lu que pour audit.

In [ ]:
!pip -q install pyarrow pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, sqlite3, time
from collections import defaultdict
import pyarrow.parquet as pq

ROOT = Path('/content/drive/MyDrive/BugPredictor/defectors_v2')
RAW_DIR = ROOT / 'raw' / 'line_bug_prediction_splits' / 'time'
AUDIT_DIR = ROOT / 'audit'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS = ('train', 'val', 'test')
BATCH_SIZE = 256

### Vérifier les fichiers et leurs colonnes

In [ ]:
paths = {s: RAW_DIR / f'{s}.parquet.gzip' for s in SPLITS}
for split, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'{path} absent. Exécutez d’abord le notebook 01.')
    pf = pq.ParquetFile(path)
    print(f'\n{split}: {pf.metadata.num_rows:,} lignes, {pf.metadata.num_row_groups} row groups')
    print(pf.schema)
    required = {'content', 'lines', 'repo', 'filepath'}
    # ParquetFile.schema.names may list physical leaf names instead of the logical top-level List field names.
    # schema_arrow.names returns the top-level logical fields (including `lines`).
    arrow_columns = set(pf.schema_arrow.names)
    print('Colonnes de premier niveau (Arrow):', sorted(arrow_columns))
    missing = required - arrow_columns
    if missing:
        raise ValueError(f'Colonnes requises absentes dans {split}: {missing}. Colonnes Arrow: {sorted(arrow_columns)}')

### Audit exhaustif par lots
La borne des lignes est vérifiée en comptant les lignes du contenu décodé en UTF-8 avec remplacement des octets invalides. Les résultats servent au contrôle de qualité; le champ `lines` reste le label publié et la cible fichier est dérivée seulement comme `1` si la liste est non vide, `0` sinon.

In [ ]:
db_path = '/content/bug_predictor_overlap_audit.sqlite'
conn = sqlite3.connect(db_path)
conn.execute('CREATE TABLE IF NOT EXISTS content_hashes (digest TEXT NOT NULL, split TEXT NOT NULL, PRIMARY KEY(digest, split))')
conn.execute('CREATE TABLE IF NOT EXISTS repo_paths (repo TEXT NOT NULL, filepath TEXT NOT NULL, split TEXT NOT NULL, PRIMARY KEY(repo, filepath, split))')

summary = {}
for split, path in paths.items():
    stats = defaultdict(int)
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=['content', 'lines', 'repo', 'filepath']):
        rows = batch.to_pylist()
        hash_rows, path_rows = [], []
        for row in rows:
            stats['rows'] += 1
            content, labels = row.get('content'), row.get('lines')
            repo, filepath = row.get('repo'), row.get('filepath')
            if content is None:
                stats['content_null'] += 1
                if labels is None:
                    stats['labels_null'] += 1
                elif len(labels) == 0:
                    stats['negative_empty_label'] += 1
                else:
                    stats['positive_nonempty_label'] += 1
                    stats['positive_with_null_content'] += 1
            else:
                stats['non_null_content_rows'] += 1
                raw = content.encode('utf-8') if isinstance(content, str) else bytes(content)
                digest = hashlib.sha256(raw).hexdigest()
                hash_rows.append((digest, split))
                text = raw.decode('utf-8', errors='replace')
                line_count = len(text.splitlines())
                stats['utf8_replacement_rows'] += int('\ufffd' in text)
                if labels is None:
                    stats['labels_null'] += 1
                elif len(labels) == 0:
                    stats['negative_empty_label'] += 1
                else:
                    stats['positive_nonempty_label'] += 1
                    stats['positive_with_null_content'] += 0
                    for value in labels:
                        if not isinstance(value, int):
                            stats['label_noninteger_values'] += 1
                        elif value < 1 or value > line_count:
                            stats['label_out_of_bounds_values'] += 1
            if labels is None and content is None:
                stats['null_content_and_label'] += 1
            if repo is not None and filepath is not None:
                path_rows.append((str(repo), str(filepath), split))
        conn.executemany('INSERT OR IGNORE INTO content_hashes VALUES (?, ?)', hash_rows)
        conn.executemany('INSERT OR IGNORE INTO repo_paths VALUES (?, ?, ?)', path_rows)
        conn.commit()
    summary[split] = dict(stats)
    print(split, dict(stats))

content_overlap = conn.execute("SELECT COUNT(*) FROM (SELECT digest FROM content_hashes GROUP BY digest HAVING COUNT(DISTINCT split) > 1)").fetchone()[0]
path_overlap = conn.execute("SELECT COUNT(*) FROM (SELECT repo, filepath FROM repo_paths GROUP BY repo, filepath HAVING COUNT(DISTINCT split) > 1)").fetchone()[0]
within_split_duplicates = {}
for split in SPLITS:
    unique = conn.execute('SELECT COUNT(*) FROM content_hashes WHERE split=?', (split,)).fetchone()[0]
    non_null_content = summary[split].get('non_null_content_rows', 0)
    within_split_duplicates[split] = non_null_content - unique
conn.close()

In [ ]:
report = {
    'dataset_doi': '10.5281/zenodo.7708984',
    'partition_family': 'line_bug_prediction_splits/time',
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'per_split': summary,
    'exact_content_hashes_in_multiple_splits': content_overlap,
    'repo_filepath_pairs_in_multiple_splits': path_overlap,
    'duplicate_content_rows_within_each_split': within_split_duplicates,
    'target_derivation': 'file_defective = 1 if lines is non-empty else 0; labels null must be investigated, not silently mapped',
    'policy': 'test partition is audit-only and must not be used for fitting, tuning, calibration, or threshold selection'
}
report_path = AUDIT_DIR / 'time_split_audit.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Rapport:', report_path)

### Lire les résultats avec prudence
Comparez les distributions observées aux attentes sans supposer que les proportions du petit échantillon test sont universelles. Des doublons exacts ou des chemins `(repo, filepath)` présents dans plusieurs partitions signalent un risque de fuite à traiter avant l’évaluation. Une valeur de ligne hors bornes ou un label nul doit être investigué et documenté; ne corrigez pas silencieusement le dataset.